<a href="https://colab.research.google.com/github/yuweimin2077-hub/xsim-chip/blob/main/notebooks/04_slice_wise_3d_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Connect 2D CT reconstruction to 3D defect inspection

**Run all cells on a T4 GPU**, or use the automatic CPU fallback. This small experiment connects the existing modules:

**3D candidate → separate projections for every slice → FBP for every slice → reconstructed 3D attenuation → fixed-threshold material labels → the existing 3D inspector → volume/position report and independent truth evaluation.**

The 16 × 129 × 129 reference is an extruded package cross-section. Defects occupy different z intervals, so the candidate varies with depth. Every slice is projected/reconstructed separately with seed 2060 + z; no reconstructed slice is duplicated to manufacture depth.

The simplified parallel rays stay within each XY slice (no cone divergence or z mixing). Detector rows align with slices at 16 µm isotropic spacing. Reference and candidate share coordinates; real scanner calibration and registration are outside this demo.

In [ ]:
from pathlib import Path
repo = Path('/content/xsim-chip')
if repo.exists():
    !git -C {repo} pull --ff-only
else:
    !git clone -q https://github.com/yuweimin2077-hub/xsim-chip.git {repo}
%cd /content/xsim-chip
%pip install -q -e '.[simulation]'

In [ ]:
import json
import time
import astra
import matplotlib.pyplot as plt
import numpy as np
from xsim_chip_analysis import (
    SimulationConfig, build_run_manifest, make_demo_volume, material_to_attenuation,
    simulate_parallel_projection, reconstruct_projection_stack,
    inspect_reconstructed_volume, evaluate_volume_defects,
)

DEPTH, SIZE = 16, 129
config = SimulationConfig(
    profile='slice-wise-parallel-3d',
    volume_shape_zyx=(DEPTH, SIZE, SIZE), detector_shape_rc=(DEPTH, 192),
    projections=180, voxel_size_um=16.0, seed=2060,
)
use_cuda = bool(astra.use_cuda())
print('ASTRA:', astra.__version__, '| CUDA:', use_cuda)
if use_cuda:
    print(astra.get_gpu_info())
reference_volume, candidate_truth_volume, defect_truth = make_demo_volume(DEPTH, SIZE, config.seed)
print('Actual volume:', reference_volume.shape)
print('Ground-truth voxels:', {kind: int(mask.sum()) for kind, mask in defect_truth.items()})

## 1 Generate the measured projection stack

Only the simulator sees candidate truth. At each angle the assembled detector image has **16 rows × 192 bins**. The saved acquisition has shape **(180 angles, 16 z rows, 192 bins)**. Relative attenuation is multiplied by 0.016 mm before Beer–Lambert projection and Poisson sampling.

In [ ]:
pipeline_started = time.perf_counter()
projection_started = time.perf_counter()
sinograms, acquisition_records = [], []
for z in range(DEPTH):
    sinogram, acquisition = simulate_parallel_projection(
        material_to_attenuation(candidate_truth_volume[z]),
        pixel_size_um=config.voxel_size_um, views=config.projections,
        detector_bins=config.detector_shape_rc[1], photons=100_000,
        seed=config.seed + z, use_cuda=use_cuda,
    )
    sinograms.append(sinogram)
    acquisition_records.append({'z_index': z, **acquisition})
projection_stack = np.stack(sinograms, axis=1)
projection_seconds = time.perf_counter() - projection_started
assert projection_stack.shape == (config.projections, DEPTH, config.detector_shape_rc[1])
print('Saved projection layout (angle, z, bin):', projection_stack.shape)

## 2 Reconstruct using only projections and geometry

This function has no reference, candidate-label, or defect-truth arguments. It applies the same ASTRA Hann-filtered FBP used by Notebook 02 to **each detector row's own sinogram**, then stacks the slices in z order.

In [ ]:
reconstruction_started = time.perf_counter()
reconstructed_volume, reconstruction_records = reconstruct_projection_stack(
    projection_stack, size=SIZE, voxel_size_um=config.voxel_size_um, use_cuda=use_cuda,
)
reconstruction_seconds = time.perf_counter() - reconstruction_started
assert reconstructed_volume.shape == reference_volume.shape
print('Reconstructed attenuation (z, y, x):', reconstructed_volume.shape)
print('Reconstructed slices:', len(reconstruction_records))

## 3 Send reconstructed material labels to the existing 3D inspector

Reuse fixed thresholds 0.075, 0.40, 0.80. The stacked labels are passed to `analyze_against_reference`, which returns positions, µm³ volumes, severity, and process hypotheses. Keep components with at least eight voxels using **26-neighbour 3D connectivity**.

The general report covers six material-difference signatures. Evaluation below scores only the three injected signatures (missing solder, excess solder, missing copper). The count of material-difference components is not the number of true physical defects.

In [ ]:
inspection_started = time.perf_counter()
segmented_volume, predicted_masks, inspection_report = inspect_reconstructed_volume(
    reference_volume, reconstructed_volume,
    voxel_size_um=config.voxel_size_um, min_component_voxels=8,
)
inspection_seconds = time.perf_counter() - inspection_started
pipeline_seconds = time.perf_counter() - pipeline_started

# Truth is introduced only now, for evaluation; it cannot change predictions.
evaluation = evaluate_volume_defects(predicted_masks, defect_truth, voxel_size_um=config.voxel_size_um)
truth_attenuation = material_to_attenuation(candidate_truth_volume)
nrmse = float(np.sqrt(np.mean(((truth_attenuation - reconstructed_volume) / 0.95)**2)))
inspection_report['evaluation'] = evaluation
print('3D material-difference report:', json.dumps(inspection_report['summary'], indent=2))
print('Strict 3D voxel-overlap metrics:', json.dumps(evaluation['union'], indent=2))
for kind, metrics in evaluation['per_defect'].items():
    print(kind, json.dumps(metrics))

## 4 Inspect the data connection and reconstruction errors

The top row shows the same z = 8 plane at three stages. The lower row shows a measured projection and depth-resolved defect masks. **Z display is expanded for visibility**; axes give voxel coordinates. Green = true positive, red = false positive, blue = missed voxel.

The depth profiles retain all false positives, including boundary errors on clean slices. High missing-solder volume error is expected from the simple threshold baseline; root-cause entries remain screening hypotheses.

In [ ]:
z_show = DEPTH // 2
predicted_union = np.logical_or.reduce(list(predicted_masks.values()))
truth_union = np.logical_or.reduce(list(defect_truth.values()))
tp = predicted_union & truth_union
fp = predicted_union & ~truth_union
fn = ~predicted_union & truth_union

overview = plt.figure(figsize=(15, 9))
for index, (image, title) in enumerate([
    (reference_volume[z_show], f'Known-good reference | z={z_show}'),
    (reconstructed_volume[z_show], f'Reconstruction | z={z_show}'),
    (segmented_volume[z_show], f'Segmented labels | z={z_show}'),
], start=1):
    ax = overview.add_subplot(2, 3, index)
    ax.imshow(image, cmap='gray', vmin=0, vmax=.95 if index == 2 else 255)
    ax.set_title(title)
    ax.axis('off')
ax = overview.add_subplot(2, 3, 4)
ax.imshow(projection_stack[60], cmap='gray', aspect='auto')
ax.set_title('One projection | 16 rows x 192 bins')
ax.set_xlabel('Detector bin')
ax.set_ylabel('Detector row = z slice')

def scatter_mask(ax, mask, color, label):
    positions = np.argwhere(mask)
    ax.scatter(positions[:, 2], positions[:, 1], positions[:, 0], s=3, alpha=.55, c=color, label=label)

ax_truth = overview.add_subplot(2, 3, 5, projection='3d')
for kind, color in zip(defect_truth, ['tab:blue', 'tab:orange', 'tab:purple']):
    scatter_mask(ax_truth, defect_truth[kind], color, kind)
ax_truth.set_title('True defects at different depths')
ax_truth.legend(loc='upper left', fontsize=7)
ax_pred = overview.add_subplot(2, 3, 6, projection='3d')
for mask, color, label in [(tp, 'green', 'TP'), (fp, 'red', 'FP'), (fn, 'blue', 'FN')]:
    scatter_mask(ax_pred, mask, color, label)
ax_pred.set_title('3D detection errors')
ax_pred.legend(loc='upper left', fontsize=8)
for ax in (ax_truth, ax_pred):
    ax.set(xlabel='x voxel', ylabel='y voxel', zlabel='z voxel',
           xlim=(0, SIZE), ylim=(0, SIZE), zlim=(0, DEPTH))
    ax.set_box_aspect((1, 1, .65))
overview.suptitle(
    f"Connected slice-wise 3D | Precision={evaluation['union']['precision']:.3f} | "
    f"Recall={evaluation['union']['recall']:.3f} | Dice={evaluation['union']['dice']:.3f}"
)
overview.tight_layout()

depth_figure, axes = plt.subplots(1, 3, figsize=(14, 3.6))
for ax, kind in zip(axes, defect_truth):
    ax.plot(np.arange(DEPTH), defect_truth[kind].sum(axis=(1, 2)), 'o-', label='Truth')
    ax.plot(np.arange(DEPTH), predicted_masks[kind].sum(axis=(1, 2)), 's--', label='Detected')
    ax.set(title=kind, xlabel='z slice', ylabel='Voxels per slice')
    ax.legend()
depth_figure.suptitle('Each depth uses its own projection and reconstruction')
depth_figure.tight_layout();

## 5 Save the connected experiment

Save the measured projection stack, reconstructed attenuation, segmented volume, known-good reference, truth labels/masks, predicted masks, 3D report, per-layer timing/seed metadata, and figures. The same inspector can re-read the two saved label volumes through `xsim-inspect`.

This is an idealized **slice-wise parallel-beam volume** demonstration. Its result does not establish cone-beam reconstruction, real-data material segmentation, calibrated defect volumes, or proven process causes.

In [ ]:
output_dir = Path('/content/xsim_volume_outputs')
output_dir.mkdir(parents=True, exist_ok=True)
manifest = build_run_manifest(config, elapsed_seconds=pipeline_seconds)
manifest['experiment'] = {
    'method': 'slice_wise_parallel_beam_3d',
    'volume_shape_zyx': list(reconstructed_volume.shape),
    'projection_shape_angle_z_bin': list(projection_stack.shape),
    'voxel_size_um': config.voxel_size_um,
    'seed_by_slice': [config.seed + z for z in range(DEPTH)],
    'reconstructed_slices': len(reconstruction_records),
    'reference_model': 'extruded_package_with_depth_bounded_candidate_defects',
    'geometry_assumptions': 'parallel rays, zero z mixing, aligned reference, isotropic spacing',
    'memory_estimate_scope': 'declared buffers only; not measured peak memory',
}
manifest['result'] = {
    'algorithm': reconstruction_records[0]['algorithm'],
    'normalized_rmse': nrmse, 'normalization': 'shared_known_attenuation_max_0.95',
    'zero_count_rays': sum(row['zero_count_rays'] for row in acquisition_records),
    'timings_seconds': {
        'projection_noise': projection_seconds,
        'reconstruction_wall': reconstruction_seconds,
        'fbp_run_and_readback_sum': sum(row['fbp_run_and_readback_seconds'] for row in reconstruction_records),
        'segmentation_and_3d_inspection': inspection_seconds,
        'acquisition_reconstruction_inspection': pipeline_seconds,
    },
    'inspection_summary': inspection_report['summary'], 'defect_evaluation': evaluation,
}
manifest['acquisition_by_slice'] = acquisition_records
manifest['reconstruction_by_slice'] = reconstruction_records
for filename, array in {
    'reference_labels.npy': reference_volume, 'candidate_truth_labels.npy': candidate_truth_volume,
    'projection_stack_angle_z_bin.npy': projection_stack,
    'reconstructed_attenuation.npy': reconstructed_volume,
    'segmented_labels.npy': segmented_volume,
}.items():
    np.save(output_dir / filename, array)
np.savez_compressed(output_dir / 'predicted_defect_masks.npz', **predicted_masks)
np.savez_compressed(output_dir / 'ground_truth_defect_masks.npz', **defect_truth)
(output_dir / 'run_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
(output_dir / 'volume_inspection_report.json').write_text(json.dumps(inspection_report, indent=2), encoding='utf-8')
overview.savefig(output_dir / 'volume_connection.jpg', dpi=160, bbox_inches='tight')
depth_figure.savefig(output_dir / 'depth_profiles.jpg', dpi=160, bbox_inches='tight')
print(json.dumps(manifest['result'], indent=2))
print('Saved to', output_dir)